# dscraft.graph quickstart

This notebook demonstrates `dscraft.graph`'s sparse tensor adapter (`PyGSparseAdapter`, a real COO<->CSR/CSC bridge between PyTorch Geometric and `scipy.sparse`) plus a minimal `GCN` forward pass.

**Section 1** builds a small synthetic Erdos-Renyi random graph, converts formats through the adapter, and runs a GCN forward pass over random node features.

**Section 2** repeats the exact same adapter/GCN pipeline against a real graph -- `torch_geometric.datasets.KarateClub` (Zachary's Karate Club), generated in-process from a hardcoded edge list bundled inside `torch_geometric` (no network access, no download).

This mirrors `packages/dscraft/examples/graph/gcn_example.py` in notebook form.

Requires the `graph` extra:
```bash
pip install -e "packages/dscraft[graph]"
```

In [1]:
import random
import warnings

# tqdm.auto emits a benign TqdmWarning at import time (via torch_geometric's
# own import of tqdm.auto) when ipywidgets isn't installed -- it just means
# tqdm falls back to its plain-text progress bar; it does not affect the
# adapter/GCN correctness demonstrated below. Suppressed narrowly (this
# category only), matching dscraft.forecast's own narrowly-scoped warning
# suppression pattern around known-benign third-party warnings.
from tqdm.std import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

import torch
from torch_geometric.datasets import KarateClub

from dscraft.graph import GCN, PyGSparseAdapter, resolve_device


def make_erdos_renyi_edge_index(num_nodes: int, p: float, seed: int = 0) -> torch.Tensor:
    """Build a small directed Erdos-Renyi random graph's edge_index."""
    rng = random.Random(seed)
    src: list[int] = []
    dst: list[int] = []
    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j and rng.random() < p:
                src.append(i)
                dst.append(j)
    if not src:
        src = list(range(num_nodes))
        dst = [(i + 1) % num_nodes for i in range(num_nodes)]
    return torch.tensor([src, dst], dtype=torch.long)


device = resolve_device()
print(f"Using device: {device}")

Using device: mps


## Section 1: synthetic Erdos-Renyi graph

In [2]:
num_nodes = 12
in_channels = 6
hidden_channels = 16
out_channels = 4

edge_index = make_erdos_renyi_edge_index(num_nodes, p=0.2, seed=42)
print(f"Synthetic Erdos-Renyi graph: {num_nodes} nodes, {edge_index.shape[1]} directed edges")

adapter = PyGSparseAdapter.from_edge_index(edge_index, num_nodes=num_nodes)
print(f"Adapter native format: {adapter.native_format}, shape: {adapter.shape}")

csr_adapter = adapter.to_csr()
print(f"Converted to CSR via scipy.sparse: nnz={csr_adapter.scipy_matrix.nnz}")

csc_adapter = adapter.to_csc()
print(f"Converted to CSC via scipy.sparse: nnz={csc_adapter.scipy_matrix.nnz}")

torch.manual_seed(42)
node_features = torch.randn(num_nodes, in_channels, device=device)

model = GCN(in_channels, hidden_channels, out_channels).to(device)
model.eval()

with torch.no_grad():
    output = model(node_features, csr_adapter)

print(f"GCN forward pass output shape: {tuple(output.shape)}")
assert output.shape == (num_nodes, out_channels)
assert torch.isfinite(output).all(), "GCN produced non-finite output"
print("GCN forward pass produced finite output of the expected shape.")

Synthetic Erdos-Renyi graph: 12 nodes, 26 directed edges
Adapter native format: coo, shape: (12, 12)
Converted to CSR via scipy.sparse: nnz=26
Converted to CSC via scipy.sparse: nnz=26


GCN forward pass output shape: (12, 4)
GCN forward pass produced finite output of the expected shape.


## Section 2: real graph -- torch_geometric.datasets.KarateClub

In [3]:
dataset = KarateClub()
data = dataset[0]

print(
    f"Real KarateClub graph: {data.num_nodes} nodes, {data.num_edges} "
    f"directed edges, {data.num_node_features} node features, "
    f"{dataset.num_classes} classes (ground-truth factions)"
)

real_adapter = PyGSparseAdapter.from_edge_index(data.edge_index, num_nodes=data.num_nodes)
print(f"Adapter native format: {real_adapter.native_format}, shape: {real_adapter.shape}")

real_csr_adapter = real_adapter.to_csr()
real_csc_adapter = real_adapter.to_csc()
print(
    f"Converted to CSR via scipy.sparse: nnz={real_csr_adapter.scipy_matrix.nnz}; "
    f"converted to CSC via scipy.sparse: nnz={real_csc_adapter.scipy_matrix.nnz}"
)

x = data.x.to(device)
real_model = GCN(
    in_channels=data.num_node_features,
    hidden_channels=16,
    out_channels=dataset.num_classes,
).to(device)
real_model.eval()

with torch.no_grad():
    real_output = real_model(x, real_csr_adapter)

print(f"GCN forward pass output shape: {tuple(real_output.shape)}")
assert real_output.shape == (data.num_nodes, dataset.num_classes)
assert torch.isfinite(real_output).all(), "GCN produced non-finite output"
print("GCN forward pass produced finite output of the expected shape on the real graph.")

Real KarateClub graph: 34 nodes, 156 directed edges, 34 node features, 4 classes (ground-truth factions)
Adapter native format: coo, shape: (34, 34)
Converted to CSR via scipy.sparse: nnz=156; converted to CSC via scipy.sparse: nnz=156
GCN forward pass output shape: (34, 4)
GCN forward pass produced finite output of the expected shape on the real graph.


## Summary

This notebook exercised `dscraft.graph`'s `PyGSparseAdapter` COO<->CSR/CSC bridge and a two-layer `GCN` forward pass, first on a synthetic Erdos-Renyi graph and then on the real, bundled `KarateClub` dataset -- confirming finite, correctly-shaped output in both cases.

See `packages/dscraft/README.md`'s `## dscraft.graph` section for more detail.